In [9]:
import numpy as np
import itertools
from typing import Optional

In [64]:
class Stratification:
    def __init__(self, name: str, strata: list[str]):
        self.name = name
        self.strata = strata

    def __repr__(self):
        return f"Stratification: {self.name}"

In [159]:
class Compartment:
    def __init__(self, strata, index):
        self.strata = strata
        self.index = index

    def __repr__(self):
        return "Compartment :" + repr(self.strata)

class CompartmentMap:
    def __init__(self, base_strat: Stratification):
        self.compartments = [Compartment([(base_strat,s)],i) for i,s in enumerate(base_strat.strata)]
        self.stratifications: dict[Stratification, Optional[tuple]] = {base_strat: None}
        #self.strat_index_map = {
        #    (base_strat,stratum):np.array([i]) for i, stratum in enumerate(base_strat.strata) 
        #}
    
    def stratify(self, strat: Stratification, stratifies: tuple[Stratification,str]):
        for existing_strat, estrat_stratifies in self.stratifications.items():
            if existing_strat.name == strat.name:
                print("Existing")
                print(estrat_stratifies, stratifies)
                #+++ Actually check for overlap, not just equivalency
                if estrat_stratifies == stratifies:
                    raise Exception("Existing stratification with same name overlaps", strat.name, stratifies)
        out_comps = []
        i = 0
        for c in self.compartments:
            if stratifies in c.strata:
                for stratum in strat.strata:
                    new_c = Compartment(c.strata + [(strat, stratum)], i)
                    out_comps.append(new_c)
                    i += 1
            else:
                out_comps.append(c)
                i += 1

        self.compartments = np.array(out_comps)
        self.stratifications[strat] = stratifies

        return strat

In [164]:
pop_strat = Stratification("pop", ["wolf","human"])
sm = CompartmentMap(pop_strat)

In [165]:
wolf_class_strat = sm.stratify(Stratification("wolf_class", ["A","B"]),(pop_strat,"wolf"))
disease_state_strat = sm.stratify(Stratification("disease_state", ["S","I","R"]), (pop_strat, "human"))
h_age_strat = sm.stratify(Stratification("age", ["child","adult","older"]), (pop_strat, "human"))
work_risk_strat = sm.stratify(Stratification("work_risk", ["low", "high"]), (h_age_strat, "adult"))
w_age_strat = sm.stratify(Stratification("age", ["juvenile","adult"]), (pop_strat, "wolf"))
severity_strat = sm.stratify(Stratification("severity", ["asymp", "mild", "severe"]), (disease_state_strat, "I"))

Existing
(Stratification: pop, 'human') (Stratification: pop, 'wolf')


In [166]:
sm.compartments

array([Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'adult'), (Stratification: work_risk, 'low')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'adult'), (Stratification: work_risk, 'high')],
       Compartment :[(Stratification: pop, 'human'), (Stratification:

In [ ]:
flow_graph; infection
unique  unique         per_src_comp
irate * (i/totalpop) * src_comp_val

predation

prate * wolf_pop * src_comp_val => cap_max_predation

In [ ]:
Flow:
    src_compartments
    dest_compartments
    flow_value()

In [162]:
sm.compartments

array([Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'older')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'I'), (Stratification: age, 'child'), (Stratification: se

In [ ]:
# Flow broadcasting
# Should this be a property of flows, or stratifications, or something else?

# infection: S->I
# 
# S->[I_asymp, I_mild, I_severe]
# Direct adjustment (split by "stratification population")
# Multi_adjustment; older populations end up in severe more often
# Do we disregard strat population altogether? Have an agexseverity transition table?
# If we stratify first, then adjustment makes more sense "with the flow"
# If we set the flow first, this doesn't make sense

In [187]:
list(itertools.permutations((a,b,c,d)))

[(array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 1, 0, 1, 0, 1, 0, 2]),
  array([0, 0, 0, 0, 1, 1, 2, 2]),
  array([0, 1, 2, 0, 1, 2, 0, 1])),
 (array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 1, 0, 1, 0, 1, 0, 2]),
  array([0, 1, 2, 0, 1, 2, 0, 1]),
  array([0, 0, 0, 0, 1, 1, 2, 2])),
 (array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 0, 0, 0, 1, 1, 2, 2]),
  array([0, 1, 0, 1, 0, 1, 0, 2]),
  array([0, 1, 2, 0, 1, 2, 0, 1])),
 (array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 0, 0, 0, 1, 1, 2, 2]),
  array([0, 1, 2, 0, 1, 2, 0, 1]),
  array([0, 1, 0, 1, 0, 1, 0, 2])),
 (array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 1, 2, 0, 1, 2, 0, 1]),
  array([0, 1, 0, 1, 0, 1, 0, 2]),
  array([0, 0, 0, 0, 1, 1, 2, 2])),
 (array(['a', 'a', 'a', 'a', 'b', 'b', 'b', 'c'], dtype='<U1'),
  array([0, 1, 2, 0, 1, 2, 0, 1]),
  array([0, 0, 0, 0, 1, 1, 2, 2]),
  array([0, 1, 0

array([0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3])

In [212]:
a = np.arange(16)
b = np.arange(3)
c = np.arange(9)
d = np.arange(256)

len(list(itertools.product(a,b,c,d)))

#6groups, each 2mul; 12mul
#8groups, each 3mul, 19mul

110592

In [ ]:
infection: agegroup


In [214]:
import jax
from jax import numpy as jnp

In [218]:
a.reshape((a.shape[0],1))

array([[ 0],
       [ 1],
       [ 2],
       [ 3],
       [ 4],
       [ 5],
       [ 6],
       [ 7],
       [ 8],
       [ 9],
       [10],
       [11],
       [12],
       [13],
       [14],
       [15]])

In [222]:
ar = a.reshape((a.shape[0],1))*1.1
dr = d.reshape((1,d.shape[0]))*0.1

jax.make_jaxpr(jnp.multiply)(ar,dr)

{ lambda ; a:f32[16,1] b:f32[1,256]. let c:f32[16,256] = mul a b in (c,) }

In [182]:
np.unique(list(zip(a,b)),axis=0)

array([['a', '0'],
       ['a', '1'],
       ['b', '0'],
       ['b', '1'],
       ['c', '2']], dtype='<U21')

In [ ]:
class TransitionFlow:
    def __init__(self, name, src_query, dst_query):
        

In [131]:
# Stratifications: 

In [147]:
def cmap_to_ctable(cmap: CompartmentMap):
    strat_compartment_indices = {}
    strat_comp_strata = {}

    

    for i,c in enumerate(cmap.compartments):
        for (strat, stratum) in c.strata:
            #print(c,strat,stratum)
            strat_comp_idx_l = strat_compartment_indices.setdefault(strat, [])
            strat_comp_strata_l = strat_comp_strata.setdefault(strat, [])
            strat_comp_idx_l.append(i)
            strat_comp_strata_l.append(strat.strata.index(stratum))
    
    strat_compartment_indices = {k: np.array(v,dtype=int) for k,v in strat_compartment_indices.items()}
    strat_comp_strata = {k: np.array(v,dtype=int) for k,v in strat_comp_strata.items()}

    return strat_compartment_indices, strat_comp_strata

In [148]:
sci, scs = cmap_to_ctable(sm)
sci, scs

({Stratification: pop: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
         17, 18]),
  Stratification: wolf_class: array([0, 1, 2, 3]),
  Stratification: age: array([0, 1, 2, 3]),
  Stratification: disease_state: array([ 4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18]),
  Stratification: age: array([ 4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18]),
  Stratification: severity: array([ 7,  8,  9, 10, 11, 12, 13, 14, 15])},
 {Stratification: pop: array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  Stratification: wolf_class: array([0, 0, 1, 1]),
  Stratification: age: array([0, 1, 0, 1]),
  Stratification: disease_state: array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2]),
  Stratification: age: array([0, 1, 2, 0, 0, 0, 1, 1, 1, 2, 2, 2, 0, 1, 2]),
  Stratification: severity: array([0, 1, 2, 0, 1, 2, 0, 1, 2])})

In [140]:
sm.compartments[sci[wolf_class_strat][scs[wolf_class_strat] == 1]]

array([Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'adult')]],
      dtype=object)

In [136]:
sm.compartments

array([Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'A'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'juvenile')],
       Compartment :[(Stratification: pop, 'wolf'), (Stratification: wolf_class, 'B'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'S'), (Stratification: age, 'older')],
       Compartment :[(Stratification: pop, 'human'), (Stratification: disease_state, 'I'), (Stratification: age, 'child')],
       Compartment

In [ ]:
# something like numpy axes, but not quite...
# mapping tables 
# "age_compartments": [indices of all compartments stratified by age]
# "age_comp_strata": [values of which age stratum said compartments contain]

In [ ]:
class CompartmentData:
    def __init__(self, cmap, data):
        self.cmap = cmap
        self.data = data